# Phase 12 - Die Adjektiv-Sonde: den Praediktor messen statt raten

**Braucht eine A100**, ~30 min.

Zwei von Hand kodierte Achsen sind gefallen - die erste post hoc, die zweite ausserhalb
der Stichprobe widerlegt (Spearman −0.04, p = 0.56). Was bleibt, ist ein grosser,
reproduzierter Effekt **ohne Theorie**: 26 Adjektive an derselben Stelle, fast alle ein
Token, Kipprate 0.0 % bis 67.2 %. Die Kalibrier-Anker haben sich zwischen zwei Laeufen
reproduziert, die Spreizung ist also kein Rauschen.

Diese Zelle raet nicht mehr. Sie fragt: **ist die Kipprate eine Funktion der
Repraesentation des Adjektivs?**

## Der Schnitt ist historisch, nicht von mir gewaehlt

* **ALT** — 26 Adjektive, deren Rate in frueheren Laeufen schon gemessen wurde
* **NEU** — 38 Adjektive, die noch nie gelaufen sind

**H1 (primaer)** passt eine Ridge-Regression **nur auf die 26 alten** an und sagt die
38 neuen voraus. Diesen Test kann ich nicht durch geschickte Wortwahl bestehen: der
Schnitt stand fest, bevor die Zelle geschrieben wurde, und fuer die 38 neuen gibt es
keine Vorhersage von mir.

## Vier Darstellungen - damit auch das *Wo* beantwortet wird

`emb` (Eingabe-Einbettung, reine Wortidentitaet) ist **vorregistriert als primaer**, weil
sie keine willkuerliche Schichtwahl braucht. `L11`, `L23`, `L35` (Residuum an der
Adjektiv-Position im echten Prompt) sind sekundaer mit Bonferroni ueber 3. Sagt erst der
Kontext voraus und nicht die Wortidentitaet, ist die Schicht, in der es zuerst
funktioniert, der Ort, an dem die Groesse gebildet wird.

**H3 (Kontrolle)** laesst einen trivialen Merkmalssatz mitlaufen — Wortlaenge und
Tokenzahl. Sagt der genauso gut voraus, traegt die Einbettung nichts Eigenes.

## Ein Fallstrick, der im Test aufgefallen ist

Leave-one-out korreliert auf **reinem Rauschen systematisch negativ** (~−0.4 bei n = 12):
die Vorhersage fuer Punkt i ist im Grenzfall der Mittelwert der uebrigen und damit
gegenlaeufig. Ein Nulltest, der nur die Paarung mischt, haette ein rho von 0
faelschlich als Nullbefund gelesen. H2 durchlaeuft deshalb je Permutation das ganze
Leave-one-out neu.

Faellt H1, ist die Rate keine glatte Funktion der Wortdarstellung — dann haben
benachbarte Woerter im Raum weit auseinanderliegende Raten, und die Frage lautet nicht
mehr *welche Dimension*, sondern *warum ist es keine Dimension*. Das ist ein
vorgesehener Ausgang mit eigenem Verdikt.


In [ ]:
# === PHASE 12 - DIE ADJEKTIV-SONDE: DEN PRAEDIKTOR MESSEN, NICHT RATEN =====
# Zwei von Hand kodierte Achsen sind gefallen. Die erste war post hoc, die
# zweite ausserhalb der Stichprobe widerlegt (Spearman -0.04, p=0.56). Was
# BLEIBT, ist ein grosser, reproduzierter Effekt ohne Theorie: 26 Adjektive an
# derselben Stelle, fast alle ein Token, Kipprate 0.0% bis 67.2%. Die vier
# Kalibrier-Anker haben sich zwischen zwei Laeufen reproduziert (p >= 0.08),
# die Spreizung ist also kein Rauschen.
#
# Diese Zelle raet nicht mehr. Sie fragt das Modell selbst: ist die Kipprate
# eine Funktion der REPRAESENTATION des Adjektivs? Dafuer wird eine Ridge-
# Regression von der Wortdarstellung auf die gemessene Rate angepasst - und
# zwar so, dass ich den Test nicht durch Wortwahl bestehen kann.
#
# DER ENTSCHEIDENDE SCHNITT IST HISTORISCH, NICHT VON MIR GEWAEHLT:
#   ALT  26 Adjektive, deren Rate in frueheren Laeufen schon gemessen wurde
#   NEU  38 Adjektive, die noch nie gelaufen sind
# H1 passt NUR auf die 26 alten an und sagt die 38 neuen voraus. Der Schnitt
# stand fest, bevor diese Zelle geschrieben wurde.
#
# VIER DARSTELLUNGEN, damit auch die Frage WO beantwortet wird:
#   emb  Eingabe-Einbettung des Adjektiv-Tokens (reine Wortidentitaet)
#   L11 / L23 / L35  Residuum an der Adjektiv-Position im echten Prompt
# VORREGISTRIERT ist emb als primaer - es ist die einzige Darstellung, die
# keine willkuerliche Schichtwahl braucht. Die drei Schichten sind sekundaer
# mit Bonferroni ueber 3.
#
# EIN TRIVIALER MERKMALSSATZ laeuft mit: Wortlaenge in Zeichen und Tokenzahl.
# Sagt der genauso gut voraus, traegt die Einbettung nichts Eigenes.
#
# VORAB REGISTRIERT:
#   H1 (primaer)  Ridge auf den 26 ALTEN angepasst sagt die 38 NEUEN mit
#                 positivem Spearman voraus. Permutationstest (Ziel der
#                 Trainingsdaten gemischt, voll neu angepasst), alpha 0.05.
#   H2            Leave-one-out ueber alle 64 hat positiven Spearman.
#   H3 (Kontrolle) Der triviale Merkmalssatz sagt SCHLECHTER voraus als emb.
# Faellt H1, ist die Rate keine glatte Funktion der Wortdarstellung, und die
# Frage lautet anders. Das ist ein vorgesehener Ausgang.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig). "
                       "Laufzeit -> Sitzung neu starten, dann NUR diese Zelle."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_sonde")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
# die 26 schon gemessenen - der Schnitt ist Geschichte, nicht meine Wahl
ALT=["precise","particular","specific","designated","exact","applicable","actual",
     "correct","given","individual","relevant","respective","proper","corresponding",
     "equivalent","canonical","customary","matching","standard","associated",
     "verbatim","printed","written","literal","own","native"]
# 38 neue, nie gelaufen. Keine Vorhersage - das ist der Punkt.
NEU=["usual","common","typical","normal","regular","ordinary","general","main",
     "primary","principal","current","existing","established","recognized",
     "accepted","preferred","assigned","listed","stated","displayed","published",
     "registered","formal","popular","familiar","everyday","traditional","modern",
     "alternative","secondary","short","abbreviated","complete","true","real",
     "genuine","authentic","appropriate"]
WOERTER=ALT+NEU
def phrase_mit(adj):
    return PHRASE if not adj else "each service's %s local name"%adj
def setze_arm(text,neu):
    if text.count(PHRASE)!=1: return text,False
    return text.replace(PHRASE,neu),True
def logit(p,eps=1e-6):
    p=min(max(p,eps),1-eps); return math.log(p/(1-p))
def ziel(k,n):
    """Glaettung nach Haldane-Anscombe, damit 0/n und n/n endlich bleiben"""
    return logit((k+0.5)/(n+1.0))
# ---- Ridge in der dualen Form: n Beispiele << d Dimensionen ---------------
def ridge_fit(X,y,alpha):
    mx=X.mean(0); my=float(y.mean()); Xc=X-mx; yc=y-my
    K=Xc@Xc.T
    a=np.linalg.solve(K+alpha*np.eye(len(y)),yc)
    return dict(mx=mx,my=my,Xc=Xc,a=a)
def ridge_pred(m,Xn):
    return (Xn-m["mx"])@m["Xc"].T@m["a"]+m["my"]
def waehle_alpha(X,y,alphas):
    """inneres Leave-one-out, explizit gerechnet - exakt statt Naeherung"""
    best=(None,float("inf"))
    for al in alphas:
        s=0.0
        for i in range(len(y)):
            tr=[j for j in range(len(y)) if j!=i]
            m=ridge_fit(X[tr],y[tr],al)
            s+=(ridge_pred(m,X[i:i+1])[0]-y[i])**2
        if s<best[1]: best=(al,s)
    return best[0]
def loo_vorhersage(X,y,alphas):
    out=np.zeros(len(y))
    for i in range(len(y)):
        tr=[j for j in range(len(y)) if j!=i]
        al=waehle_alpha(X[tr],y[tr],alphas)
        out[i]=ridge_pred(ridge_fit(X[tr],y[tr],al),X[i:i+1])[0]
    return out
# ---- Rangstatistik ---------------------------------------------------------
def raenge(v):
    idx=sorted(range(len(v)),key=lambda i:v[i]); r=[0.0]*len(v); i=0
    while i<len(idx):
        j=i
        while j+1<len(idx) and v[idx[j+1]]==v[idx[i]]: j+=1
        m=(i+j)/2.0+1.0
        for k in range(i,j+1): r[idx[k]]=m
        i=j+1
    return r
def pearson(x,y):
    n=len(x); mx=sum(x)/n; my=sum(y)/n
    sxy=sum((a-mx)*(b-my) for a,b in zip(x,y))
    sx=math.sqrt(sum((a-mx)**2 for a in x)); sy=math.sqrt(sum((b-my)**2 for b in y))
    return sxy/(sx*sy) if sx*sy else float("nan")
def spearman(x,y): return pearson(raenge(list(x)),raenge(list(y)))
def bestimmtheit(y,yh):
    y=np.asarray(y,float); yh=np.asarray(yh,float)
    ss=float(((y-yh)**2).sum()); st=float(((y-y.mean())**2).sum())
    return 1.0-ss/st if st>0 else float("nan")
def haltefeld(Xa,ya,Xn,yn,alphas,perm=2000,startwert=20260805):
    """H1: nur auf ALT anpassen, NEU vorhersagen. Permutation mischt die
       TRAININGS-Ziele und passt vollstaendig neu an - ein gueltiger Nulltest."""
    al=waehle_alpha(Xa,ya,alphas)
    yh=ridge_pred(ridge_fit(Xa,ya,al),Xn)
    rho=spearman(yh,yn)
    rnd=random.Random(startwert); yy=list(ya); tr=0
    for _ in range(perm):
        rnd.shuffle(yy)
        yh2=ridge_pred(ridge_fit(Xa,np.array(yy),al),Xn)
        if spearman(yh2,yn)>=rho: tr+=1
    return dict(alpha=float(al),rho=float(rho),p=(tr+1)/(perm+1.0),
                r2=float(bestimmtheit(yn,yh)),vorhersage=[float(v) for v in yh])
def loo_vorbereiten(X,alpha):
    """Je Fold alles vorrechnen, was NICHT von y abhaengt. Damit kostet eine
       Permutation nur noch ein Skalarprodukt je Fold statt einer Anpassung."""
    n=len(X); vor=[]
    for i in range(n):
        tr=[j for j in range(n) if j!=i]
        Xt=X[tr]; mx=Xt.mean(0); Xc=Xt-mx
        G=np.linalg.inv(Xc@Xc.T+alpha*np.eye(n-1))
        vor.append((tr,((X[i]-mx)@Xc.T)@G))
    return vor
def loo_mit(vor,y):
    out=np.zeros(len(y))
    for i,(tr,kg) in enumerate(vor):
        yt=y[tr]; my=float(yt.mean()); out[i]=float(kg@(yt-my))+my
    return out
def perm_loo(X,y,alpha,perm=400,startwert=20260805):
    """H2-Nulltest, RICHTIG gerechnet: y wird gemischt und das ganze
       Leave-one-out neu durchlaufen. Das ist noetig, weil LOO auf reinem
       Rauschen systematisch NEGATIV korreliert (die Vorhersage fuer Punkt i
       ist im Grenzfall der Mittelwert der uebrigen und damit gegenlaeufig).
       Ein Nulltest, der nur die Paarung mischt, haette diesen Versatz
       uebersehen und ein rho von 0 faelschlich als Nullbefund gelesen."""
    vor=loo_vorbereiten(X,alpha)
    rho=spearman(loo_mit(vor,y),y)
    rnd=random.Random(startwert); yy=list(y); tr=0
    for _ in range(perm):
        rnd.shuffle(yy); ya=np.array(yy)
        if spearman(loo_mit(vor,ya),ya)>=rho: tr+=1
    return float(rho),(tr+1)/(perm+1.0),float(np.mean([0.0]))
def urteil_sonde(H1,H2,H3,alpha_p=0.05,alpha_s=0.05/3):
    """emb ist primaer; die Schichten sekundaer mit Bonferroni ueber 3."""
    # "trivial reicht" nur, wenn der triviale Satz auch WIRKLICH vorhersagt -
    # sonst wuerde ein emb mit NEGATIVEM rho faelschlich hierher fallen
    if (H3 is not None and H1 is not None and H3["rho"]>0 and H3["p"]<alpha_p
            and H3["rho"]>=H1["rho"]):
        return "TRIVIAL-REICHT"
    if H1 is not None and H1["p"]<alpha_p and H1["rho"]>0: return "EINBETTUNG-SAGT-VORAUS"
    kontext=[k for k,v in (H2 or {}).items() if v and v["p"]<alpha_s and v["rho"]>0]
    if kontext: return "KONTEXT-SAGT-VORAUS"
    return "KEINE-VORHERSAGE"
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250
            and any(a<=ord(c)<=b for a,b in FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_ARM=int(globals().get("N_ARM",48)); MAX_NEW=int(globals().get("MAX_NEW",64))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260805))
SCHICHTEN=[11,23,35]; ALPHAS=[1e0,1e1,1e2,1e3,1e4,1e5,1e6]
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
BASIS=PROMPTS[ZIEL_ID]; assert BASIS.count(PHRASE)==1
assert not (set(ALT)&set(NEU)), "ALT und NEU ueberschneiden sich"
assert len(set(WOERTER))==len(WOERTER), "Wort doppelt"
print("="*82)
print("ADJEKTIV-SONDE | %d Woerter (%d alt, %d neu) + Original | %d Ziehungen"
      %(len(WOERTER),len(ALT),len(NEU),N_ARM))
print("="*82)
print("Der Schnitt ALT/NEU ist historisch: ALT wurde in frueheren Laeufen schon")
print("gemessen, NEU nie. H1 passt NUR auf ALT an und sagt NEU voraus.")
TEXTE={"original":BASIS}
for w in WOERTER:
    t,ok=setze_arm(BASIS,phrase_mit(w)); assert ok,"Arm %s nicht baubar"%w
    TEXTE[w]=t
NAMEN=["original"]+WOERTER
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
# ---- Darstellungen einsammeln (ein Vorwaertspass je Wort, ohne Sampling) ----
print("")
print("DARSTELLUNGEN einsammeln (Eingabe-Einbettung + Residuum an der Adjektiv-Position)")
EMB=model.get_input_embeddings().weight
REP={"emb":{}}; REP.update({"L%d"%l:{} for l in SCHICHTEN})
TRIV={}
for w in WOERTER:
    ids=tokenizer(TEXTE[w],add_special_tokens=False)["input_ids"]
    st=[tokenizer.decode([i]) for i in ids]
    j=st.index(" local")                       # das Adjektiv steht direkt davor
    tid=tokenizer(" "+w,add_special_tokens=False)["input_ids"]
    REP["emb"][w]=EMB[torch.tensor(tid,device=EMB.device)].float().mean(0).detach().cpu().numpy()
    TRIV[w]=np.array([len(w),len(tid)],dtype=float)
    with torch.no_grad():
        o=model(torch.tensor([ids],device=model.device),output_hidden_states=True)
    for l in SCHICHTEN:
        REP["L%d"%l][w]=o.hidden_states[l+1][0,j-1].float().cpu().numpy()
    del o
gc.collect(); torch.cuda.empty_cache()
print("  fertig: %d Woerter x (%d + %d Darstellungen)"%(len(WOERTER),1,len(SCHICHTEN)))
# ---- Erzeugung --------------------------------------------------------------
print("")
K={}; KB={}; N={}; CLB={}; ROH={}
t0=time.time()
for ai,nm in enumerate(NAMEN):
    txt=prompt_text(TEXTE[nm]); ant=[]
    for b0 in range(0,N_ARM,CHUNK):
        b=min(CHUNK,N_ARM-b0)
        enc=tokenizer([txt]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(SEED+1009*ai+b0)
        with torch.no_grad():
            gen=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                               repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                               pad_token_id=tokenizer.pad_token_id)
        for j2 in range(b):
            ant.append(tokenizer.decode(gen[j2,enc["input_ids"].shape[1]:],
                                        skip_special_tokens=True))
    ROH[nm]=ant
    cb=[classify_breit(a) for a in ant]; CLB[nm]=collections.Counter(cb)
    K[nm]=sum(1 for a in ant if classify_answer(a) in SW)
    KB[nm]=sum(1 for c in cb if c in SWB); N[nm]=len(ant)
    if ai%8==0 or ai==len(NAMEN)-1:
        print("  [%2d/%2d] %-14s %5.1f%%   (%.0f s)"
              %(ai+1,len(NAMEN),nm,100*KB[nm]/N[nm],time.time()-t0))
# ---- Ziele ------------------------------------------------------------------
Y={w:ziel(KB[w],N[w]) for w in WOERTER}
print("")
print("GEMESSENE RATEN, sortiert (o = schon frueher gemessen):")
for w in sorted(WOERTER,key=lambda x:KB[x]/N[x]):
    p,lo,hi=wilson(KB[w],N[w])
    print("  %s %-14s %2d/%-3d = %5.1f%% [%4.1f,%4.1f]"
          %("o" if w in ALT else " ",w,KB[w],N[w],100*p,100*lo,100*hi))
print("  %s %-14s %2d/%-3d = %5.1f%%   <- ohne Adjektiv"
      %(" ","original",KB["original"],N["original"],100*KB["original"]/N["original"]))
# ---- H1: nur auf ALT anpassen, NEU vorhersagen ------------------------------
ya=np.array([Y[w] for w in ALT]); yn=np.array([Y[w] for w in NEU])
def matrix(rep,ws): return np.stack([rep[w] for w in ws]).astype(np.float64)
print("")
print("H1 - ANGEPASST NUR AUF DIE %d ALTEN, VORHERGESAGT DIE %d NEUEN"%(len(ALT),len(NEU)))
print("  %-8s %8s %8s %10s %10s"%("Darst.","Spearman","R2","Permut. p","alpha"))
H1={}; H1S={}
for nm in ["emb"]+["L%d"%l for l in SCHICHTEN]:
    r=haltefeld(matrix(REP[nm],ALT),ya,matrix(REP[nm],NEU),yn,ALPHAS)
    (H1 if nm=="emb" else H1S)[nm]=r
    print("  %-8s %+8.3f %8.3f %10.4f %10.0e"%(nm,r["rho"],r["r2"],r["p"],r["alpha"]))
H3=haltefeld(matrix(TRIV,ALT),ya,matrix(TRIV,NEU),yn,ALPHAS)
print("  %-8s %+8.3f %8.3f %10.4f %10.0e   <- Kontrolle: nur Laenge und Tokenzahl"
      %("trivial",H3["rho"],H3["r2"],H3["p"],H3["alpha"]))
# ---- H2: Leave-one-out ueber alle 64 ----------------------------------------
print("")
print("H2 - LEAVE-ONE-OUT ueber alle %d Woerter"%len(WOERTER))
yall=np.array([Y[w] for w in WOERTER])
H2={}
for nm in ["emb"]+["L%d"%l for l in SCHICHTEN]:
    Xn=matrix(REP[nm],WOERTER)
    yh=loo_vorhersage(Xn,yall,ALPHAS)
    al=waehle_alpha(Xn,yall,ALPHAS)   # fuer den Nulltest ein festes alpha
    rho,p,_=perm_loo(Xn,yall,al)
    H2[nm]=dict(rho=float(rho),p=float(p),alpha=float(al),
                r2=float(bestimmtheit(yall,yh)),vorhersage=[float(v) for v in yh])
    print("  %-8s Spearman %+.3f   R2 %+.3f   Permut. p %.4f"%(nm,rho,H2[nm]["r2"],p))
# ---- Verdikt ----------------------------------------------------------------
CODE=urteil_sonde(H1.get("emb"),H1S,H3)
print("")
print("VERDIKT: %s"%CODE)
if CODE=="EINBETTUNG-SAGT-VORAUS":
    print("  Eine Ridge-Regression, die NUR die 26 alten Woerter gesehen hat, sagt")
    print("  die Rangfolge von 38 nie gelaufenen Woertern voraus. Damit ist die")
    print("  Kipprate eine glatte Funktion der Wortdarstellung - und die Richtung")
    print("  im Einbettungsraum ist ein GEMESSENES Objekt, keine Vermutung.")
elif CODE=="KONTEXT-SAGT-VORAUS":
    print("  Die reine Wortidentitaet reicht nicht, das Residuum im Prompt schon.")
    print("  Dann entsteht die Groesse erst im Kontext - genau die Schicht, in der")
    print("  sie zuerst vorhersagt, ist der Ort, an dem sie gebildet wird.")
elif CODE=="TRIVIAL-REICHT":
    print("  Wortlaenge und Tokenzahl sagen mindestens so gut voraus wie die")
    print("  Einbettung. Dann ist der Effekt nicht semantisch, und die 2048")
    print("  Dimensionen haben nur die zwei trivialen Merkmale nachgebildet.")
else:
    print("  Keine Darstellung sagt die neuen Woerter voraus. Die Rate ist dann")
    print("  keine glatte Funktion des Wortes: benachbarte Woerter im Raum haben")
    print("  weit auseinanderliegende Raten. Die Frage lautet dann anders - nicht")
    print("  'welche Dimension', sondern 'warum ist es keine Dimension'.")
print("")
print("H1 ist der Test. H2 misst dieselbe Sache mit mehr Daten, teilt sich aber")
print("Trainings- und Testwoerter - deshalb ist es sekundaer. H3 ist die Kontrolle.")
print("Zu H2: ein negatives Spearman ist dort KEIN Gegenbefund. Leave-one-out")
print("korreliert auf reinem Rauschen systematisch negativ (~-0.4 bei n=12); der")
print("Nulltest bildet das ab, indem er das ganze LOO je Permutation neu durchlaeuft.")
if H1.get("emb"):
    ordn=sorted(zip(NEU,H1["emb"]["vorhersage"],[KB[w]/N[w] for w in NEU]),
                key=lambda x:-x[1])
    print("")
    print("DIE 38 NEUEN, nach VORHERSAGE sortiert (nicht nach Messung):")
    print("  %-14s %10s %10s"%("Wort","vorherges.","gemessen"))
    for w,v,g in ordn: print("  %-14s %10.2f %9.1f%%"%(w,v,100*g))
SONDE_RESULTS=dict(verdict=CODE,prompt_id=ZIEL_ID,n_arm=N_ARM,max_new=MAX_NEW,temp=TEMP,
    seed=SEED,alt=ALT,neu=NEU,schichten=SCHICHTEN,alphas=ALPHAS,
    k_streng=K,k_breit=KB,n=N,ziel={w:Y[w] for w in WOERTER},
    klassen_breit={n_:dict(CLB[n_]) for n_ in CLB},
    H1=dict(H1,**H1S),H2=H2,H3=H3)
wc_save("antworten_sonde",dict(prompt_id=ZIEL_ID,prompts=TEXTE,antworten=ROH))
np.savez_compressed(os.path.join(RUN_OUT,"darstellungen.npz"),
                    woerter=np.array(WOERTER),
                    **{("%s_%s"%(r,w)):REP[r][w] for r in REP for w in WOERTER})
wc_save_all()
print("")
print("(%d Woerter x %d Ziehungen, Temperatur %.2f, %d neue Token, Denken aus."
      %(len(WOERTER),N_ARM,TEMP,MAX_NEW))
print(" Alle Texte und alle Darstellungen liegen in Drive - die Regression laesst")
print(" sich damit offline beliebig wiederholen, ohne die GPU noch einmal zu brauchen.)")
